# Testing for Late Fusion Techniques

### Here we aim to combine the discriminative powers of:

1. Structural Features (FUSE embeddings)
2. Semantic Features (Word-2-Vec) [provided in the OGBN benchmark dataset]

### Two techniques were tested:

1. Simple Concatenation
2. Contrastive Alignment (CLIP-based)

## Results Summary


| Split Type | Fusion Technique | Classifier | Test Accuracy | Weighted Avg F1 Score |
|---|---|---|---:|---:|
| OGBN Official Split | Simple Concatenation | MLP | **0.7029** | **0.6870** |
| OGBN Official Split | Simple Concatenation | Multinomial Logistic Regression | **0.4760** | **0.4281** |
| OGBN Official Split | CLIP-based Contrastive Alignment | MLP | **0.6941** | **0.6792** |
| OGBN Official Split | CLIP-based Contrastive Alignment | Multinomial Logistic Regression | **0.7633** | **0.7558** |
| 30-70 Split | Simple Concatenation | MLP | **0.8498** | **0.8409** |
| 30-70 Split | Simple Concatenation | Multinomial Logistic Regression | **0.7910** | **0.7772** |
| 30-70 Split | CLIP-based Contrastive Alignment | MLP | **0.8722** | **0.8654** |
| 30-70 Split | CLIP-based Contrastive Alignment | Multinomial Logistic Regression | **0.8552** | **0.8489** |


## Interpretation


- Simple concatenation directly combines the two feature types without explicitly aligning their embedding geometries. Therefore, it may require a more expressive nonlinear classifier such as an MLP to extract useful interactions between structural and semantic information.

- In contrast, CLIP-based contrastive alignment aims to learn a shared representation space where the two feature types are geometrically aligned. If this learned representation is effective, it should become more linearly separable.

Thus, better performance with multinomial logistic regression indicates that the CLIP-based fusion method produces a cleaner and more discriminative embedding.




In [1]:
!pip install ogb
!pip install torch_geometric

import torch
import numpy as np
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import to_torch_csr_tensor   

import torch
from ogb.nodeproppred import NodePropPredDataset 


import wandb 
from sklearn.linear_model import LogisticRegression

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 88.0 MB/s eta 0:00:00:00:01:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.

In [2]:

original_torch_load = torch.load

def patched_torch_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return original_torch_load(*args, **kwargs)

torch.load = patched_torch_load

In [3]:
# loading the pre-saved dataset indices:

train_idx = torch.load(r"/kaggle/input/datasets/mehulgoyal1729/ogbn-products-metadataokayy/Products_ogbn_train_index")
test_idx= torch.load(r"/kaggle/input/datasets/mehulgoyal1729/ogbn-products-metadataokayy/Products_ogbn_test_index")
valid_idx = torch.load(r"/kaggle/input/datasets/mehulgoyal1729/ogbn-products-metadataokayy/Products_ogbn_valid_index")
labels_tensor = torch.load(r"/kaggle/input/datasets/mehulgoyal1729/ogbn-products-metadataokayy/Products_ogbn_labels_tensor")
num_classes = 47
node_features = torch.load(r"/kaggle/input/datasets/mehulgoyal1729/ogbn-products-metadataokayy/Products_ogbn_node_features")

torch.cuda.empty_cache()

## Testing with Simple Concatenation:

In [4]:
S = torch.load(r"/kaggle/input/datasets/mehulgoyal1729/s-products-normed-k100/S_products_normed_k100")
S_given = node_features

In [5]:
node_features.shape

torch.Size([2449029, 100])

### MLP:

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy
from torch.utils.data import TensorDataset, DataLoader

S = S.to(device)
S_given = S_given.to(device)

S_generated = torch.cat((S, S_given) , dim = 1)
del S, S_given

print(f"Shape of concatenated embeddings: {S_generated.shape}")

train_idx = torch.tensor(train_idx).to(device)
valid_idx =  torch.tensor(valid_idx).to(device)
test_idx  =  torch.tensor(test_idx).to(device)
labels_tensor = torch.tensor(labels_tensor).to(device) 


S_train = S_generated[train_idx].detach()
y_train = labels_tensor[train_idx].long()

S_valid = S_generated[valid_idx].detach()
y_valid = labels_tensor[valid_idx].long()

S_test  = S_generated[test_idx].detach()
y_test  = labels_tensor[test_idx].long()
del S_generated
print(f"Train nodes: {S_train.shape[0]} | Valid nodes: {S_valid.shape[0]} | Test nodes: {S_test.shape[0]}")

mean = S_train.mean(dim=0, keepdim=True)
std  = S_train.std(dim=0, keepdim=True) + 1e-7

S_train_scaled = (S_train - mean) / std
S_valid_scaled = (S_valid - mean) / std
S_test_scaled  = (S_test - mean) / std

batch_size = 2048
train_dataset = TensorDataset(S_train_scaled, y_train)
train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), 
            nn.ReLU(),
            nn.Dropout(0.5), 
            nn.Linear(hidden, hidden), 
            nn.ReLU(),
            nn.Dropout(0.5), 
            nn.Linear(hidden, out_dim)
        )
    def forward(self, x): return self.net(x)

K = S_train.shape[1]
model = MLP(K, 512, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 200
losses = []
best_val_acc = 0.0
best_model_weights = None
patience = 20  
epochs_no_improve = 0
target_threshold = 0.99

for epoch in range(epochs):
    
    model.train()
    epoch_loss = 0.0
    
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    avg_train_loss = epoch_loss / len(train_loader)
    losses.append(epoch_loss)
    
    model.eval()
    with torch.no_grad():
        val_logits = model(S_valid_scaled)
        val_preds = val_logits.argmax(dim=1)
        val_correct = (val_preds == y_valid).sum().item()
        val_acc = val_correct / len(y_valid)

    print(f'Epoch {epoch+1:3d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Acc: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_weights = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if val_acc >= target_threshold:
        print(f"\nTarget Validation Accuracy ({target_threshold}) reached! Stopping early.")
        break
        
    if epochs_no_improve >= patience:
        print(f"\nEarly stopping triggered! No improvement for {patience} epochs.")
        break

print(f"\nLoading best model weights (Val Acc: {best_val_acc:.4f}) for Final Testing...")
model.load_state_dict(best_model_weights)


del S_train, S_valid

test_loader = DataLoader(
    TensorDataset(S_test_scaled, y_test),
    batch_size=4096,
    shuffle=False
)
from sklearn.metrics import classification_report, accuracy_score, f1_score

model.eval()

all_preds = []

with torch.no_grad():
    for batch_x, _ in test_loader:
        batch_x = batch_x.to(device)
        logits = model(batch_x)
        preds = logits.argmax(dim=1)
        all_preds.append(preds.cpu())

test_preds = torch.cat(all_preds)
test_correct = (test_preds == y_test.cpu()).sum().item()
test_acc = test_correct / len(y_test)

print(f"Final Test Accuracy: {test_acc:.4f}")
print(f"Final OGBN-Products Test Accuracy: {test_acc:.4f}")
print("\n--- MLP Classification Report ---")
print(classification_report(
    y_test.detach().cpu().numpy(),
    test_preds.detach().cpu().numpy(),
    digits=4
))


Shape of concatenated embeddings: torch.Size([2449029, 200])


/tmp/ipykernel_58/2423232819.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels_tensor = torch.tensor(labels_tensor).to(device)


Train nodes: 196615 | Valid nodes: 39323 | Test nodes: 2213091
Epoch   1/200 | Train Loss: 0.3441 | Val Acc: 0.8615
Epoch   2/200 | Train Loss: 0.0069 | Val Acc: 0.8565
Epoch   3/200 | Train Loss: 0.0029 | Val Acc: 0.8556
Epoch   4/200 | Train Loss: 0.0015 | Val Acc: 0.8552
Epoch   5/200 | Train Loss: 0.0015 | Val Acc: 0.8509
Epoch   6/200 | Train Loss: 0.0010 | Val Acc: 0.8536
Epoch   7/200 | Train Loss: 0.0003 | Val Acc: 0.8543
Epoch   8/200 | Train Loss: 0.0004 | Val Acc: 0.8554
Epoch   9/200 | Train Loss: 0.0004 | Val Acc: 0.8568
Epoch  10/200 | Train Loss: 0.0005 | Val Acc: 0.8504
Epoch  11/200 | Train Loss: 0.0003 | Val Acc: 0.8528
Epoch  12/200 | Train Loss: 0.0005 | Val Acc: 0.8511
Epoch  13/200 | Train Loss: 0.0004 | Val Acc: 0.8521
Epoch  14/200 | Train Loss: 0.0005 | Val Acc: 0.8555
Epoch  15/200 | Train Loss: 0.0005 | Val Acc: 0.8486
Epoch  16/200 | Train Loss: 0.0008 | Val Acc: 0.8513
Epoch  17/200 | Train Loss: 0.0005 | Val Acc: 0.8493
Epoch  18/200 | Train Loss: 0.0008 |

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0     0.6033    0.7441    0.6664    100670
           1     0.6224    0.7722    0.6893     95236
           2     0.8442    0.8438    0.8440    102870
           3     0.6467    0.6993    0.6720    135255
           4     0.8388    0.8469    0.8428    596195
           5     0.6625    0.8681    0.7515     35985
           6     0.6905    0.7533    0.7205    140356
           7     0.8366    0.9207    0.8766    152202
           8     0.7051    0.9397    0.8057     97679
           9     0.8685    0.8534    0.8609     59852
          10     0.6452    0.7626    0.6990     47165
          11     0.0946    0.3140    0.1454     28933
          12     0.7209    0.4827    0.5782    128632
          13     0.6808    0.7751    0.7249     88694
          14     0.1268    0.3639    0.1881      2734
          15     0.7674    0.8893    0.8239     23871
          16     0.7490    0.1678    0.2742     83019
          17     0.8561    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Multinomial Logistic Regressiom:

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

S = torch.load(r"/kaggle/input/datasets/mehulgoyal1729/s-products-normed-k100/S_products_normed_k100")
S_given = node_features

S = S.to(device)
S_given = S_given.to(device)

S_generated = torch.cat((S, S_given) , dim = 1)
del S, S_given

print(f"Shape of concatenated embeddings: {S_generated.shape}")

train_idx = torch.tensor(train_idx).to(device)
valid_idx =  torch.tensor(valid_idx).to(device)
test_idx  =  torch.tensor(test_idx).to(device)
labels_tensor = torch.tensor(labels_tensor).to(device) 


S_train = S_generated[train_idx].detach()
y_train = labels_tensor[train_idx].long()

S_valid = S_generated[valid_idx].detach()
y_valid = labels_tensor[valid_idx].long()

S_test  = S_generated[test_idx].detach()
y_test  = labels_tensor[test_idx].long()



print("\n--- Transferring Data to CPU for Scikit-Learn ---")
X_train_np = S_train.cpu().numpy()
y_train_np = y_train.cpu().numpy()

X_test_np  = S_test.cpu().numpy()
y_test_np  = y_test.cpu().numpy()

print("Training Logistic Regression...")
clf = LogisticRegression(max_iter=1000, multi_class='multinomial', n_jobs=-1)
clf.fit(X_train_np, y_train_np)

print("Evaluating on Test Set...")
y_pred = clf.predict(X_test_np)


print(f"Accuracy: {accuracy_score(y_pred,y_test_np )}")


print("\n--- Logistic Regression Classification Report ---")
print(classification_report(y_test_np, y_pred, digits=4))

Shape of concatenated embeddings: torch.Size([2449029, 200])

--- Transferring Data to CPU for Scikit-Learn ---


/tmp/ipykernel_58/311732516.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_idx = torch.tensor(train_idx).to(device)
/tmp/ipykernel_58/311732516.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  valid_idx =  torch.tensor(valid_idx).to(device)
/tmp/ipykernel_58/311732516.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_idx  =  torch.tensor(test_idx).to(device)
/tmp/ipykernel_58/311732516.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach(

Training Logistic Regression...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Evaluating on Test Set...
Accuracy: 0.4760025683534929

--- Logistic Regression Classification Report ---


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0     0.3262    0.3939    0.3569    100670
           1     0.3903    0.3646    0.3770     95236
           2     0.6777    0.6542    0.6657    102870
           3     0.3162    0.4082    0.3564    135255
           4     0.4942    0.8458    0.6239    596195
           5     0.4075    0.2835    0.3344     35985
           6     0.3387    0.4304    0.3791    140356
           7     0.6035    0.5145    0.5555    152202
           8     0.6775    0.7559    0.7146     97679
           9     0.6806    0.4858    0.5670     59852
          10     0.3948    0.3369    0.3636     47165
          11     0.2243    0.0919    0.1304     28933
          12     0.5580    0.1990    0.2934    128632
          13     0.4303    0.2782    0.3379     88694
          14     0.3383    0.0753    0.1232      2734
          15     0.5750    0.2066    0.3040     23871
          16     0.7185    0.0251    0.0485     83019
          17     0.6564    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## CLIP based Contrastive Alignment :

### Contrastively Optiizing embeddings:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import copy
import wandb 
from torch.utils.data import DataLoader, TensorDataset

# Encoding the Embeddings:
class Encoder(nn.Module):
    def __init__(self, input_dim=100, hidden_dim=256, latent_dim=128):
        super().__init__()
        self.encode_str = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
        )
        self.encode_sem = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
        )
        
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07)) # temp scaling 

    def forward(self, x_str, x_sem):
        z_str = self.encode_str(x_str)
        z_sem = self.encode_sem(x_sem)

        z_str = F.normalize(z_str, p=2, dim=-1)
        z_sem = F.normalize(z_sem, p=2, dim=-1)

        return z_str, z_sem



# InfoNCE Constratsive Loss :
def infonce_loss(z_str, z_sem, logit_scale):
    logits_per_str = logit_scale.exp() * torch.matmul(z_str, z_sem.t())
    logits_per_sem = logits_per_str.t() 

    batch_size = z_str.shape[0]
    labels = torch.arange(batch_size, dtype=torch.long, device=z_str.device)

    loss_str = F.cross_entropy(logits_per_str, labels)
    loss_sem = F.cross_entropy(logits_per_sem, labels)

    return (loss_str + loss_sem) / 2

S = S.to(device)
S_given = S_given.to(device)

print("\n--- Standardizing Tensors ---")
mean_S = S.mean(dim=0, keepdim=True)
std_S = S.std(dim=0, keepdim=True) + 1e-7
S_scaled = (S - mean_S) / std_S

mean_S_given = S_given.mean(dim=0, keepdim=True)
std_S_given = S_given.std(dim=0, keepdim=True) + 1e-7
S_given_scaled = (S_given - mean_S_given) / std_S_given

batch_size = 4096 
dataset = TensorDataset(S_scaled, S_given_scaled) 
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)




config = {
    "input_dim": 100,
    "hidden_dim": 512,
    "latent_dim": 128,
    "batch_size": batch_size,
    "learning_rate": 0.01,
    "weight_decay": 1e-4,
    "epochs": 1000,
    "patience": 10
}

wandb.init(
    project="fuse-ogbn-products-contrastive",
    name="latent-alignment-infonce",
    config=config
)

model = Encoder(
    input_dim=config["input_dim"], 
    hidden_dim=config["hidden_dim"], 
    latent_dim=config["latent_dim"]
).to(device)

optimizer = optim.Adam(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.9)

print("\n--- Starting Contrastive Latent Alignment ---")

best_loss = float('inf')
epochs_no_improve = 0
best_model_weights = copy.deepcopy(model.state_dict())

for epoch in range(config["epochs"]):
    model.train()
    total_loss = 0.0
    
    total_top1_acc = 0.0
    total_pos_sim = 0.0
    total_neg_sim = 0.0
    
    for batch_str, batch_sem in loader:
        optimizer.zero_grad()
        
        z_str, z_sem = model(batch_str, batch_sem)
        loss = infonce_loss(z_str, z_sem, model.logit_scale)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        
        with torch.no_grad():
            raw_sim = torch.matmul(z_str, z_sem.t())
            curr_batch_size = raw_sim.shape[0]
            labels = torch.arange(curr_batch_size, device=raw_sim.device)
            
            preds = raw_sim.argmax(dim=1)
            acc = (preds == labels).float().mean().item()
            total_top1_acc += acc
            
            pos_sim = torch.diag(raw_sim).mean().item()
            total_pos_sim += pos_sim
            
            sum_all = raw_sim.sum().item()
            sum_pos = torch.diag(raw_sim).sum().item()
            neg_sim = (sum_all - sum_pos) / (curr_batch_size * (curr_batch_size - 1))
            total_neg_sim += neg_sim
            
    
    num_batches = len(loader)
    avg_loss = total_loss / num_batches
    avg_top1 = total_top1_acc / num_batches
    avg_pos = total_pos_sim / num_batches
    avg_neg = total_neg_sim / num_batches
    
    current_temp = model.logit_scale.exp().item()
    
    scheduler.step()

    wandb.log({
        "epoch": epoch + 1,
        "InfoNCE_Loss": avg_loss,
        "Top1_Accuracy_Pct": avg_top1 * 100,
        "Pos_Similarity": avg_pos,
        "Neg_Similarity": avg_neg,
        "Temp_Scale": current_temp,
        "Learning_Rate": optimizer.param_groups[0]['lr']
    })

    if avg_loss < best_loss - 1e-3:
        best_loss = avg_loss
        epochs_no_improve = 0
        best_model_weights = copy.deepcopy(model.state_dict())
    else:
        epochs_no_improve += 1

    if (epoch + 1) % 1 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{config['epochs']} | Loss: {avg_loss:.4f} | "
              f"Top-1 Acc: {avg_top1*100:.1f}% | "
              f"Pos Sim: {avg_pos:.3f} | Neg Sim: {avg_neg:.3f} | "
              f"Temp: {current_temp:.1f}")



    if epoch == 50:
        break
    if epochs_no_improve >= config["patience"]:
        print(f"\nEarly stopping triggered! No improvement in loss for {config['patience']} epochs.")
        break

print("Contrastive Alignment Complete.")

# 
print(f"\nLoading best model weights (Loss: {best_loss:.4f}) for extraction...")
model.load_state_dict(best_model_weights)
model.eval()

final_dataset = TensorDataset(S_scaled, S_given_scaled)
final_loader = DataLoader(final_dataset, batch_size=4096, shuffle=False)

all_z_str = []
all_z_sem = []

with torch.no_grad():
    for batch_str, batch_sem in final_loader:
        batch_z_str, batch_z_sem = model(batch_str, batch_sem)

        all_z_str.append(batch_z_str.cpu())
        all_z_sem.append(batch_z_sem.cpu())

Final_Z_str = torch.cat(all_z_str, dim=0)
Final_Z_sem = torch.cat(all_z_sem, dim=0)

Z_fused = Final_Z_str + Final_Z_sem

print(f"Final Fused Embedding Matrix Shape: {Z_fused.shape}")

wandb.finish()


--- Standardizing Tensors ---



--- Starting Contrastive Latent Alignment ---
Epoch   1/1000 | Loss: 6.6290 | Top-1 Acc: 2.0% | Pos Sim: 0.529 | Neg Sim: 0.297 | Temp: 18.0
Epoch   2/1000 | Loss: 6.2198 | Top-1 Acc: 3.7% | Pos Sim: 0.447 | Neg Sim: 0.218 | Temp: 20.7
Epoch   3/1000 | Loss: 6.1434 | Top-1 Acc: 4.2% | Pos Sim: 0.418 | Neg Sim: 0.203 | Temp: 21.3
Epoch   4/1000 | Loss: 6.1094 | Top-1 Acc: 4.4% | Pos Sim: 0.404 | Neg Sim: 0.194 | Temp: 21.9
Epoch   5/1000 | Loss: 6.0913 | Top-1 Acc: 4.6% | Pos Sim: 0.397 | Neg Sim: 0.190 | Temp: 22.5
Epoch   6/1000 | Loss: 6.0832 | Top-1 Acc: 4.7% | Pos Sim: 0.392 | Neg Sim: 0.186 | Temp: 22.5
Epoch   7/1000 | Loss: 6.0751 | Top-1 Acc: 4.7% | Pos Sim: 0.391 | Neg Sim: 0.186 | Temp: 22.6
Epoch   8/1000 | Loss: 6.0708 | Top-1 Acc: 4.7% | Pos Sim: 0.389 | Neg Sim: 0.185 | Temp: 22.7
Epoch   9/1000 | Loss: 6.0667 | Top-1 Acc: 4.8% | Pos Sim: 0.389 | Neg Sim: 0.184 | Temp: 22.7
Epoch  10/1000 | Loss: 6.0625 | Top-1 Acc: 4.8% | Pos Sim: 0.388 | Neg Sim: 0.183 | Temp: 22.7
Epo

InfoNCE_Loss,█▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Learning_Rate,███████████████▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▁▁▁▁▁▁▁▁▁
Neg_Similarity,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Pos_Similarity,█▄▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Temp_Scale,▁▄▅▆▆▇▇▇▇▇▇▆▇▇▇▇▇▇▇▇▇▇▇█▇▇▇▇▇▇▇█████████
Top1_Accuracy_Pct,▁▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇███████████████████████
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
InfoNCE_Loss,6.00291
Learning_Rate,0.0081
Neg_Similarity,0.17297
Pos_Similarity,0.37364


In [ ]:
clip_fused = torch.load(r"/kaggle/input/datasets/mehulgoyal1729/clipproducts/Clip_Products.pth")

In [18]:
torch.save(Z_fused, "Clip_Products_new.pth")


In [12]:
S_generated = clip_fused

In [13]:
train_idx = torch.tensor(train_idx).to(device)
valid_idx = torch.tensor(valid_idx).to(device)
test_idx  = torch.tensor(test_idx).to(device)
labels_tensor = torch.tensor(labels_tensor).to(device)
S_generated = S_generated.to(device)



print(S_generated.device)
print(train_idx.device)

S_train = S_generated[train_idx].detach()
y_train = labels_tensor[train_idx].long()

S_valid = S_generated[valid_idx].detach()
y_valid = labels_tensor[valid_idx].long()

S_test = S_generated[test_idx].detach()
y_test = labels_tensor[test_idx].long()

/tmp/ipykernel_58/1200573675.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_idx = torch.tensor(train_idx).to(device)
/tmp/ipykernel_58/1200573675.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  valid_idx = torch.tensor(valid_idx).to(device)
/tmp/ipykernel_58/1200573675.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_idx  = torch.tensor(test_idx).to(device)
/tmp/ipykernel_58/1200573675.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().

cuda:0
cuda:0


### MLP

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy
from torch.utils.data import TensorDataset, DataLoader

S_generated = clip_fused

print(f"Shape of concatenated embeddings: {S_generated.shape}")

train_idx = torch.tensor(train_idx).to(device)
valid_idx = torch.tensor(valid_idx).to(device)
test_idx  = torch.tensor(test_idx).to(device)
labels_tensor = torch.tensor(labels_tensor).to(device)
S_generated = S_generated.to(device)



print(S_generated.device)
print(train_idx.device)

S_train = S_generated[train_idx].detach()
y_train = labels_tensor[train_idx].long()

S_valid = S_generated[valid_idx].detach()
y_valid = labels_tensor[valid_idx].long()

S_test = S_generated[test_idx].detach()
y_test = labels_tensor[test_idx].long()

del S_generated

print(
    f"Train nodes: {S_train.shape[0]} | "
    f"Valid nodes: {S_valid.shape[0]} | "
    f"Test nodes: {S_test.shape[0]}"
)

mean = S_train.mean(dim=0, keepdim=True)
std = S_train.std(dim=0, keepdim=True) + 1e-7

S_train_scaled = (S_train - mean) / std
S_valid_scaled = (S_valid - mean) / std
S_test_scaled  = (S_test - mean) / std

batch_size = 2048

train_dataset = TensorDataset(S_train_scaled, y_train)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),   
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),  
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        return self.net(x)

K = S_train.shape[1]

model = MLP(K, 512, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 200
best_val_acc = 0
best_model_weights = None

patience = 20
epochs_no_improve = 0
target_threshold = 0.99

for epoch in range(epochs):


    model.train()

    epoch_loss = 0

    for batch_x, batch_y in train_loader:

        optimizer.zero_grad()

        logits = model(batch_x)

        loss = criterion(logits, batch_y)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / len(train_loader)


    model.eval()

    with torch.no_grad():

        train_logits = model(S_train_scaled)
        train_preds = train_logits.argmax(dim=1)

        train_correct = (
            train_preds == y_train
        ).sum().item()

        train_acc = train_correct / len(y_train)

        val_logits = model(S_valid_scaled)

        val_preds = val_logits.argmax(dim=1)

        val_correct = (
            val_preds == y_valid
        ).sum().item()

        val_acc = val_correct / len(y_valid)

    gap = train_acc - val_acc

    print(
        f"Epoch {epoch+1:3d}/{epochs} | "
        f"Loss: {avg_train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Gap: {gap:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_weights = copy.deepcopy(
            model.state_dict()
        )
        epochs_no_improve = 0

    else:
        epochs_no_improve += 1

    if val_acc >= target_threshold:
        print(
            f"\nTarget validation accuracy "
            f"{target_threshold} reached."
        )
        break

    if epochs_no_improve >= patience:
        print(
            f"\nEarly stopping "
            f"(no improvement for {patience} epochs)"
        )
        break

print(
    f"\nLoading best model "
    f"(Val Acc={best_val_acc:.4f})"
)

model.load_state_dict(best_model_weights)

test_loader = DataLoader(
    TensorDataset(S_test_scaled, y_test),
    batch_size=4096,
    shuffle=False
)

model.eval()

all_preds = []

with torch.no_grad():

    for batch_x, _ in test_loader:

        logits = model(batch_x)

        preds = logits.argmax(dim=1)

        all_preds.append(
            preds.cpu()
        )

test_preds = torch.cat(all_preds)

test_acc = (
    (test_preds == y_test.cpu())
    .sum()
    .item()
    / len(y_test)
)

print(f"Final Test Accuracy: {test_acc:.4f}")

from sklearn.metrics import classification_report, confusion_matrix

from sklearn.metrics import classification_report

print("\n--- Classification Report ---")
print(
    classification_report(
        y_test.cpu().numpy(),
        test_preds.numpy(),
        digits=4
    )
)

Shape of concatenated embeddings: torch.Size([2449029, 128])


/tmp/ipykernel_58/2128991963.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_idx = torch.tensor(train_idx).to(device)
/tmp/ipykernel_58/2128991963.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  valid_idx = torch.tensor(valid_idx).to(device)
/tmp/ipykernel_58/2128991963.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_idx  = torch.tensor(test_idx).to(device)
/tmp/ipykernel_58/2128991963.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detac

cuda:0
cuda:0
Train nodes: 196615 | Valid nodes: 39323 | Test nodes: 2213091
Epoch   1/200 | Loss: 0.2907 | Train Acc: 0.9980 | Val Acc: 0.8143 | Gap: 0.1837
Epoch   2/200 | Loss: 0.0190 | Train Acc: 0.9989 | Val Acc: 0.8142 | Gap: 0.1847
Epoch   3/200 | Loss: 0.0093 | Train Acc: 0.9993 | Val Acc: 0.8126 | Gap: 0.1867
Epoch   4/200 | Loss: 0.0071 | Train Acc: 0.9996 | Val Acc: 0.8089 | Gap: 0.1907
Epoch   5/200 | Loss: 0.0049 | Train Acc: 0.9997 | Val Acc: 0.8056 | Gap: 0.1941
Epoch   6/200 | Loss: 0.0035 | Train Acc: 0.9998 | Val Acc: 0.8003 | Gap: 0.1995
Epoch   7/200 | Loss: 0.0027 | Train Acc: 0.9999 | Val Acc: 0.8008 | Gap: 0.1990
Epoch   8/200 | Loss: 0.0023 | Train Acc: 0.9999 | Val Acc: 0.7946 | Gap: 0.2052
Epoch   9/200 | Loss: 0.0019 | Train Acc: 0.9999 | Val Acc: 0.7899 | Gap: 0.2100
Epoch  10/200 | Loss: 0.0018 | Train Acc: 0.9999 | Val Acc: 0.7955 | Gap: 0.2044
Epoch  11/200 | Loss: 0.0018 | Train Acc: 0.9999 | Val Acc: 0.7995 | Gap: 0.2004
Epoch  12/200 | Loss: 0.0019 | T

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0     0.5189    0.7278    0.6059    100670
           1     0.5771    0.6909    0.6289     95236
           2     0.7516    0.8809    0.8111    102870
           3     0.5846    0.4332    0.4976    135255
           4     0.8503    0.8657    0.8579    596195
           5     0.6102    0.8429    0.7079     35985
           6     0.6960    0.5184    0.5943    140356
           7     0.7941    0.9240    0.8541    152202
           8     0.7150    0.9533    0.8171     97679
           9     0.8486    0.8324    0.8404     59852
          10     0.6882    0.6977    0.6929     47165
          11     0.3147    0.4484    0.3698     28933
          12     0.7411    0.5673    0.6427    128632
          13     0.7571    0.6517    0.7004     88694
          14     0.4075    0.4473    0.4265      2734
          15     0.5693    0.9362    0.7080     23871
          16     0.5657    0.4144    0.4784     83019
          17     0.7741    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [16]:
S_generated = clip_fused

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score



train_idx = torch.tensor(train_idx).to(device)
valid_idx =  torch.tensor(valid_idx).to(device)
test_idx  =  torch.tensor(test_idx).to(device)
labels_tensor = torch.tensor(labels_tensor).to(device) 

S_generated = S_generated.to(device)

S_train = S_generated[train_idx].detach()
y_train = labels_tensor[train_idx].long()

S_valid = S_generated[valid_idx].detach()
y_valid = labels_tensor[valid_idx].long()

S_test  = S_generated[test_idx].detach()
y_test  = labels_tensor[test_idx].long()



print("\n--- Transferring Data to CPU for Scikit-Learn ---")
X_train_np = S_train.cpu().numpy()
y_train_np = y_train.cpu().numpy()

X_test_np  = S_test.cpu().numpy()
y_test_np  = y_test.cpu().numpy()

print("Training Logistic Regression...")
clf = LogisticRegression(max_iter=1000, multi_class='multinomial', n_jobs=-1)
clf.fit(X_train_np, y_train_np)

print("Evaluating on Test Set...")
y_pred = clf.predict(X_test_np)


print(f"Accuracy: {accuracy_score(y_pred,y_test_np )}")


print("\n--- Logistic Regression Classification Report ---")
print(classification_report(y_test_np, y_pred, digits=4))

/tmp/ipykernel_58/3732168673.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_idx = torch.tensor(train_idx).to(device)
/tmp/ipykernel_58/3732168673.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  valid_idx =  torch.tensor(valid_idx).to(device)
/tmp/ipykernel_58/3732168673.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_idx  =  torch.tensor(test_idx).to(device)
/tmp/ipykernel_58/3732168673.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach(


--- Transferring Data to CPU for Scikit-Learn ---
Training Logistic Regression...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Evaluating on Test Set...
Accuracy: 0.7632804073578537

--- Logistic Regression Classification Report ---


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0     0.6772    0.7951    0.7314    100670
           1     0.6954    0.7543    0.7237     95236
           2     0.8932    0.8344    0.8628    102870
           3     0.7560    0.6189    0.6806    135255
           4     0.9289    0.8800    0.9038    596195
           5     0.6635    0.8536    0.7466     35985
           6     0.7338    0.7904    0.7611    140356
           7     0.7758    0.9588    0.8577    152202
           8     0.7841    0.9614    0.8638     97679
           9     0.8736    0.8919    0.8826     59852
          10     0.6468    0.8230    0.7243     47165
          11     0.3140    0.4978    0.3851     28933
          12     0.8487    0.6206    0.7170    128632
          13     0.7508    0.7800    0.7651     88694
          14     0.3449    0.4682    0.3972      2734
          15     0.6962    0.9353    0.7982     23871
          16     0.6737    0.6600    0.6668     83019
          17     0.8497    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## 30-70 Dataset Split Evaluation for Late Fusion

### Testing the Fusion techniques on 30-70 MCAR split :


In [18]:

seed = 42
mask_frac = 0.70 

labels_for_split = torch.as_tensor(labels_tensor).long().view(-1).cpu()
classes = torch.unique(labels_for_split)

generator_cpu = torch.Generator(device='cpu')
generator_cpu.manual_seed(seed)

mask_idx_list = []
labeled_idx_list = []

for c in classes:
    c_indices = (labels_for_split == c).nonzero(as_tuple=True)[0]

    num_in_class = c_indices.size(0)
    num_mask_c = int(mask_frac * num_in_class)

    perm = torch.randperm(num_in_class, generator=generator_cpu)

    mask_idx_list.append(c_indices[perm[:num_mask_c]])
    labeled_idx_list.append(c_indices[perm[num_mask_c:]])

test_idx_30_70 = torch.cat(mask_idx_list)
train_idx_30_70 = torch.cat(labeled_idx_list)

train_idx_30_70 = train_idx_30_70[torch.randperm(train_idx_30_70.numel(), generator=generator_cpu)]
test_idx_30_70 = test_idx_30_70[torch.randperm(test_idx_30_70.numel(), generator=generator_cpu)]

print("--- 30-70 Dataset Split ---")
print(f"Total nodes: {labels_for_split.numel()}")
print(f"Train nodes: {train_idx_30_70.numel()} ({train_idx_30_70.numel()/labels_for_split.numel():.2%})")
print(f"Test nodes:  {test_idx_30_70.numel()} ({test_idx_30_70.numel()/labels_for_split.numel():.2%})")

train_class_counts = torch.bincount(labels_for_split[train_idx_30_70], minlength=num_classes)
test_class_counts = torch.bincount(labels_for_split[test_idx_30_70], minlength=num_classes)

print(f"Classes present in train: {(train_class_counts > 0).sum().item()} / {num_classes}")
print(f"Classes present in test:  {(test_class_counts > 0).sum().item()} / {num_classes}")


--- 30-70 Dataset Split ---
Total nodes: 2449029
Train nodes: 734732 (30.00%)
Test nodes:  1714297 (70.00%)
Classes present in train: 47 / 47
Classes present in test:  46 / 47


In [ ]:


import torch
import torch.nn as nn
import torch.optim as optim
import copy
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

late_fusion_30_70_results = []

class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        return self.net(x)


def predict_in_batches(model, X, batch_size=16384):
    model.eval()
    preds = []

    with torch.no_grad():
        for start in range(0, X.shape[0], batch_size):
            end = start + batch_size
            batch_x = X[start:end].to(device)
            logits = model(batch_x)
            preds.append(logits.argmax(dim=1).cpu())

    return torch.cat(preds, dim=0)


def evaluate_late_fusion_30_70(
    S_eval,
    labels_eval,
    train_idx_eval,
    test_idx_eval,
    fusion_name,
    hidden_dim=512,
    batch_size=4096,
    epochs=200,
    patience=10,
    valid_ratio=0.10
):
    print("\n" + "=" * 80)
    print(f"30-70 Split Evaluation: {fusion_name}")
    print("=" * 80)

    S_eval = S_eval.detach().float().cpu()
    labels_eval = torch.as_tensor(labels_eval).long().view(-1).cpu()

    X_train = S_eval[train_idx_eval].detach()
    y_train = labels_eval[train_idx_eval].detach().long()

    X_test = S_eval[test_idx_eval].detach()
    y_test = labels_eval[test_idx_eval].detach().long()

    print(f"Train nodes: {X_train.shape[0]} | Test nodes: {X_test.shape[0]}")
    print(f"Embedding dimension: {X_train.shape[1]}")

    mean = X_train.mean(dim=0, keepdim=True)
    std = X_train.std(dim=0, keepdim=True) + 1e-7

    X_train_scaled = (X_train - mean) / std
    X_test_scaled = (X_test - mean) / std


    train_count = X_train_scaled.shape[0]
    internal_val_count = max(num_classes, int(valid_ratio * train_count))

    inner_perm_cpu = torch.randperm(train_count, generator=generator_cpu)
    inner_val_pos = inner_perm_cpu[:internal_val_count]
    inner_train_pos = inner_perm_cpu[internal_val_count:]

    X_inner_train = X_train_scaled[inner_train_pos]
    y_inner_train = y_train[inner_train_pos]

    X_inner_val = X_train_scaled[inner_val_pos]
    y_inner_val = y_train[inner_val_pos]

    train_dataset = TensorDataset(X_inner_train, y_inner_train)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    # ------------------------------------------------------------
    # MLP evaluation
    # ------------------------------------------------------------
    print("\n--- Training MLP on 30% train nodes ---")

    model = MLP(
        in_dim=X_train_scaled.shape[1],
        hidden=hidden_dim,
        out_dim=num_classes
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

    best_val_acc = 0.0
    best_model_weights = None
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_train_loss = epoch_loss / len(train_loader)

        val_preds = predict_in_batches(model, X_inner_val)
        val_acc = (val_preds == y_inner_val).float().mean().item()

        print(
            f"Epoch {epoch+1:3d}/{epochs} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_weights = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"\nMLP early stopping triggered after {patience} epochs without improvement.")
            break

    print(f"\nLoading best MLP weights with validation accuracy: {best_val_acc:.4f}")
    model.load_state_dict(best_model_weights)

    mlp_test_preds = predict_in_batches(model, X_test_scaled)
    mlp_test_acc = accuracy_score(y_test.numpy(), mlp_test_preds.numpy())
    mlp_test_f1 = f1_score(y_test.numpy(), mlp_test_preds.numpy(), average='weighted')

    print(f"\n--- {fusion_name}: 30-70 MLP Test Results ---")
    print(f"MLP Test Accuracy:    {mlp_test_acc:.4f}")
    print(f"MLP Weighted Test F1: {mlp_test_f1:.4f}")

    print("\n--- MLP Classification Report ---")
    print(classification_report(y_test.numpy(), mlp_test_preds.numpy(), digits=4))

    late_fusion_30_70_results.append({
        "Split": "30-70",
        "Fusion": fusion_name,
        "Classifier": "MLP",
        "Accuracy": mlp_test_acc,
        "Weighted F1": mlp_test_f1
    })

    print("\n--- Training Logistic Regression on 30% train nodes ---")

    X_train_np = X_train_scaled.numpy()
    y_train_np = y_train.numpy()

    X_test_np = X_test_scaled.numpy()
    y_test_np = y_test.numpy()

    clf = LogisticRegression(max_iter=1000, multi_class='multinomial', n_jobs=-1)
    clf.fit(X_train_np, y_train_np)

    print("Evaluating Logistic Regression on 70% test nodes...")
    logreg_pred = clf.predict(X_test_np)

    logreg_acc = accuracy_score(y_test_np, logreg_pred)
    logreg_f1 = f1_score(y_test_np, logreg_pred, average='weighted')

    print(f"\n--- {fusion_name}: 30-70 Logistic Regression Test Results ---")
    print(f"Logistic Regression Test Accuracy:    {logreg_acc:.4f}")
    print(f"Logistic Regression Weighted Test F1: {logreg_f1:.4f}")

    print("\n--- Logistic Regression Classification Report ---")
    print(classification_report(y_test_np, logreg_pred, digits=4))

    late_fusion_30_70_results.append({
        "Split": "30-70",
        "Fusion": fusion_name,
        "Classifier": "Multinomial Logistic Regression",
        "Accuracy": logreg_acc,
        "Weighted F1": logreg_f1
    })

    del X_train, X_test, X_train_scaled, X_test_scaled
    del X_inner_train, X_inner_val, y_inner_train, y_inner_val
    torch.cuda.empty_cache()

    return pd.DataFrame(late_fusion_30_70_results)


### 30-70 Evaluation: Simple Concatenation

In [ ]:

# Simple concat :

S_struct = torch.load(r"/kaggle/input/datasets/mehulgoyal1729/s-products-normed-k100/S_products_normed_k100",map_location="cpu").float()

S_semantic = node_features.detach().cpu().float()

S_simple_concat_30_70 = torch.cat((S_struct, S_semantic), dim=1)

print(f"Simple concatenated embedding shape: {S_simple_concat_30_70.shape}")

results_30_70 = evaluate_late_fusion_30_70(
    S_eval=S_simple_concat_30_70,
    labels_eval=labels_tensor,
    train_idx_eval=train_idx_30_70,
    test_idx_eval=test_idx_30_70,
    fusion_name="Simple Concatenation",
    hidden_dim=512,
    batch_size=4096,
    epochs=200,
    patience=10
)

display(results_30_70)

del S_struct, S_semantic, S_simple_concat_30_70
torch.cuda.empty_cache()


Simple concatenated embedding shape: torch.Size([2449029, 200])

30-70 Split Evaluation: Simple Concatenation
Train nodes: 734732 | Test nodes: 1714297
Embedding dimension: 200

--- Training MLP on 30% train nodes ---
Epoch   1/200 | Train Loss: 1.0214 | Val Acc: 0.8206
Epoch   2/200 | Train Loss: 0.7304 | Val Acc: 0.8276
Epoch   3/200 | Train Loss: 0.6943 | Val Acc: 0.8326
Epoch   4/200 | Train Loss: 0.6759 | Val Acc: 0.8374
Epoch   5/200 | Train Loss: 0.6641 | Val Acc: 0.8388
Epoch   6/200 | Train Loss: 0.6564 | Val Acc: 0.8403
Epoch   7/200 | Train Loss: 0.6498 | Val Acc: 0.8412
Epoch   8/200 | Train Loss: 0.6465 | Val Acc: 0.8427
Epoch   9/200 | Train Loss: 0.6424 | Val Acc: 0.8428
Epoch  10/200 | Train Loss: 0.6398 | Val Acc: 0.8431
Epoch  11/200 | Train Loss: 0.6379 | Val Acc: 0.8451
Epoch  12/200 | Train Loss: 0.6361 | Val Acc: 0.8437
Epoch  13/200 | Train Loss: 0.6335 | Val Acc: 0.8443
Epoch  14/200 | Train Loss: 0.6324 | Val Acc: 0.8452
Epoch  15/200 | Train Loss: 0.6308 | Val

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0     0.7933    0.8127    0.8029     80005
           1     0.7680    0.8284    0.7971     76882
           2     0.8960    0.9032    0.8996     81230
           3     0.7796    0.8107    0.7949    105742
           4     0.9250    0.9655    0.9448    468264
           5     0.8021    0.8664    0.8330     28500
           6     0.7988    0.8951    0.8442    111139
           7     0.9375    0.9360    0.9368    120539
           8     0.8997    0.9378    0.9183     77557
           9     0.9108    0.9036    0.9072     47150
          10     0.7594    0.8325    0.7943     36641
          11     0.6468    0.3589    0.4617     23055
          12     0.8331    0.8226    0.8278     92320
          13     0.8735    0.8180    0.8448     71078
          14     0.8562    0.3179    0.4636      2155
          15     0.8694    0.9255    0.8966     18837
          16     0.7226    0.7556    0.7387     58515
          17     0.9004    

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Evaluating Logistic Regression on 70% test nodes...

--- Simple Concatenation: 30-70 Logistic Regression Test Results ---
Logistic Regression Test Accuracy:    0.7910
Logistic Regression Weighted Test F1: 0.7772

--- Logistic Regression Classification Report ---


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0     0.6938    0.7670    0.7285     80005
           1     0.7427    0.7966    0.7687     76882
           2     0.8426    0.8827    0.8622     81230
           3     0.7328    0.7865    0.7587    105742
           4     0.8197    0.9001    0.8580    468264
           5     0.7960    0.8397    0.8173     28500
           6     0.7725    0.8776    0.8217    111139
           7     0.9160    0.9288    0.9224    120539
           8     0.8588    0.9154    0.8862     77557
           9     0.8909    0.8896    0.8903     47150
          10     0.7660    0.7974    0.7814     36641
          11     0.4861    0.1443    0.2225     23055
          12     0.7563    0.7113    0.7331     92320
          13     0.7997    0.7418    0.7697     71078
          14     0.5015    0.1568    0.2390      2155
          15     0.8477    0.9005    0.8733     18837
          16     0.6969    0.5362    0.6061     58515
          17     0.8989    

,Split,Fusion,Classifier,Accuracy,Weighted F1
0,30-70,Simple Concatenation,MLP,0.849829,0.840869
1,30-70,Simple Concatenation,Multinomial Logistic Regression,0.791039,0.777233


### 30-70 Evaluation: CLIP-based Contrastive Alignment

In [ ]:

# CLIP-based Contrastive Alignment fused embeddings

try:
    S_clip_30_70 = clip_fused.detach().cpu().float()
    print("Using existing variable: clip_fused")
except NameError:
    try:
        S_clip_30_70 = Z_fused.detach().cpu().float()
        print("Using existing variable: Z_fused")
    except NameError:
        S_clip_30_70 = torch.load(
            r"/kaggle/input/datasets/mehulgoyal1729/ogbn-products-clip-fused/Clip_Products.pth",
            map_location="cpu"
        ).float()
        print("Loaded CLIP-fused embeddings from saved Kaggle input path.")

print(f"CLIP-fused embedding shape: {S_clip_30_70.shape}")

results_30_70 = evaluate_late_fusion_30_70(
    S_eval=S_clip_30_70,
    labels_eval=labels_tensor,
    train_idx_eval=train_idx_30_70,
    test_idx_eval=test_idx_30_70,
    fusion_name="CLIP-based Contrastive Alignment",
    hidden_dim=512,
    batch_size=4096,
    epochs=200,
    patience=10
)

display(results_30_70)

del S_clip_30_70
torch.cuda.empty_cache()


Using existing variable: clip_fused
CLIP-fused embedding shape: torch.Size([2449029, 128])

30-70 Split Evaluation: CLIP-based Contrastive Alignment
Train nodes: 734732 | Test nodes: 1714297
Embedding dimension: 128

--- Training MLP on 30% train nodes ---
Epoch   1/200 | Train Loss: 0.8065 | Val Acc: 0.8591
Epoch   2/200 | Train Loss: 0.5701 | Val Acc: 0.8637
Epoch   3/200 | Train Loss: 0.5447 | Val Acc: 0.8660
Epoch   4/200 | Train Loss: 0.5337 | Val Acc: 0.8672
Epoch   5/200 | Train Loss: 0.5267 | Val Acc: 0.8681
Epoch   6/200 | Train Loss: 0.5221 | Val Acc: 0.8691
Epoch   7/200 | Train Loss: 0.5188 | Val Acc: 0.8693
Epoch   8/200 | Train Loss: 0.5159 | Val Acc: 0.8701
Epoch   9/200 | Train Loss: 0.5137 | Val Acc: 0.8703
Epoch  10/200 | Train Loss: 0.5110 | Val Acc: 0.8710
Epoch  11/200 | Train Loss: 0.5112 | Val Acc: 0.8709
Epoch  12/200 | Train Loss: 0.5089 | Val Acc: 0.8707
Epoch  13/200 | Train Loss: 0.5073 | Val Acc: 0.8707
Epoch  14/200 | Train Loss: 0.5063 | Val Acc: 0.8714
E

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0     0.8057    0.8405    0.8227     80005
           1     0.8040    0.8229    0.8133     76882
           2     0.9010    0.9284    0.9145     81230
           3     0.8022    0.8269    0.8144    105742
           4     0.9518    0.9768    0.9641    468264
           5     0.8487    0.8604    0.8545     28500
           6     0.8287    0.8978    0.8618    111139
           7     0.9462    0.9512    0.9487    120539
           8     0.9307    0.9380    0.9344     77557
           9     0.9235    0.9169    0.9201     47150
          10     0.7934    0.8394    0.8158     36641
          11     0.6037    0.5093    0.5525     23055
          12     0.8390    0.8800    0.8590     92320
          13     0.8890    0.8326    0.8599     71078
          14     0.8037    0.4863    0.6060      2155
          15     0.8758    0.9379    0.9058     18837
          16     0.7941    0.8328    0.8130     58515
          17     0.9144    

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Evaluating Logistic Regression on 70% test nodes...

--- CLIP-based Contrastive Alignment: 30-70 Logistic Regression Test Results ---
Logistic Regression Test Accuracy:    0.8552
Logistic Regression Weighted Test F1: 0.8489

--- Logistic Regression Classification Report ---


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0     0.7866    0.8210    0.8034     80005
           1     0.7891    0.8010    0.7950     76882
           2     0.8967    0.9185    0.9074     81230
           3     0.7612    0.8084    0.7841    105742
           4     0.9404    0.9704    0.9552    468264
           5     0.8252    0.8487    0.8368     28500
           6     0.8181    0.8756    0.8459    111139
           7     0.9433    0.9444    0.9439    120539
           8     0.9187    0.9363    0.9274     77557
           9     0.9130    0.9137    0.9134     47150
          10     0.7857    0.8125    0.7989     36641
          11     0.5901    0.4414    0.5050     23055
          12     0.8278    0.8400    0.8338     92320
          13     0.8733    0.8223    0.8470     71078
          14     0.7703    0.4979    0.6048      2155
          15     0.8824    0.9144    0.8981     18837
          16     0.7706    0.7763    0.7734     58515
          17     0.9101    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,Split,Fusion,Classifier,Accuracy,Weighted F1
0,30-70,Simple Concatenation,MLP,0.849829,0.840869
1,30-70,Simple Concatenation,Multinomial Logistic Regression,0.791039,0.777233
2,30-70,CLIP-based Contrastive Alignment,MLP,0.872219,0.865385
3,30-70,CLIP-based Contrastive Alignment,Multinomial Logistic Regression,0.855186,0.848883
